# 05e — Independent detector sanity check and repair

`05d` was not a valid transfer confirmation because the residual CNN itself was at chance level (AUC ≈ 0.502). This notebook therefore treats detector power as a prerequisite.

Protocol is fixed before inspecting the new transfer result:

1. build a repaired **EnhancedResidualCNN** using **TRAIN only** and random-allocation stegos at the already fixed teacher payload;
2. evaluate it on a disjoint TRAIN-development subset;
3. require **AUC ≥ 0.65** as a pre-specified sanity threshold;
4. only if that threshold is passed, re-run the pre-specified transfer comparison **alpha=0.25 vs alpha=1.0** on the exact common-feasible validation source IDs from `05d`;
5. do **not** access the test split and do **not** modify `frozen_allocator.json`.

The repaired network is a project-specific compact residual CNN. It must **not** be described as SRNet or XuNet.


In [ ]:
from pathlib import Path
import gc, hashlib, json, joblib, yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rdhlab.io import read_gray
from rdhlab.pipeline import run_frozen_image_precomputed
from rdhlab.blockcodec import analyze_blocks
from rdhlab.allocation import rank_blocks
from rdhlab.detectors import detector_metrics, paired_detector_bootstrap
from rdhlab.transfer import paired_method_bootstrap
from rdhlab.detector_repair import (
    train_enhanced_residual_cnn, score_enhanced_residual_cnn,
    save_enhanced_cnn,
)

config=yaml.safe_load(Path('/workspace/config/experiment.yaml').read_text())
seed=int(config['project']['seed'])
bs=int(config['dataset']['block_size'])
fixed_fpr=float(config['detectors']['fixed_fpr'])
confidence=float(config['statistics']['confidence'])
n_boot=int(config['statistics']['cluster_bootstrap_resamples'])

manifest=pd.read_csv(config['dataset']['prepared_manifest'])
train=manifest[manifest.split=='train'].reset_index(drop=True)
val=manifest[manifest.split=='validation'].reset_index(drop=True)

out=Path('/workspace/results/cnn_repair_transfer'); out.mkdir(parents=True,exist_ok=True)
models=Path('/workspace/results/models'); models.mkdir(parents=True,exist_ok=True)

teacher_validation=json.loads(Path('/workspace/results/srm_teacher_risk/teacher_validation.json').read_text())
teacher_bpp=float(teacher_validation['payload_bpp'])
local_risk=joblib.load(models/'srm_teacher_local_risk.joblib')
rule=json.loads(Path('/workspace/results/cnn_transfer/candidate_alpha_rule.json').read_text())
candidate_alpha=float(rule['candidate_alpha'])
reference_alpha=1.0
old_decision=json.loads(Path('/workspace/results/cnn_transfer/decision.json').read_text())

print('Teacher payload:',teacher_bpp)
print('Pre-specified candidate alpha:',candidate_alpha)
print('Reference alpha:',reference_alpha)
print('05d old-CNN AUC candidate/reference:',old_decision['auc_candidate'],old_decision['auc_reference'])
print('Train:',len(train),'Validation:',len(val))


## A. Fixed detector-development partition

The detector is developed entirely inside the TRAIN split. Source images are disjoint between detector training and detector development. No validation image is used to train or select the repaired detector.

Fixed configuration for this repair attempt:

- detector-train: first 2000 TRAIN source images;
- detector-dev: next 750 TRAIN source images;
- payload: 0.009 net bpp (already frozen by the earlier power-curve rule);
- allocation for detector development: random;
- epochs: 6 fixed epochs;
- paired batch size: 8 cover/stego pairs;
- sanity threshold: dev ROC-AUC ≥ 0.65.

There is no hyperparameter grid and no best-epoch selection.


In [ ]:
N_DET_TRAIN=min(2000,len(train))
N_DET_DEV=min(750,max(0,len(train)-N_DET_TRAIN))
EPOCHS=6
PAIR_BATCH_SIZE=8
SANITY_AUC_THRESHOLD=0.65

if N_DET_TRAIN < 1000 or N_DET_DEV < 300:
    raise RuntimeError('Insufficient TRAIN images for the pre-specified detector train/dev partition.')

det_train=train.iloc[:N_DET_TRAIN].reset_index(drop=True)
det_dev=train.iloc[N_DET_TRAIN:N_DET_TRAIN+N_DET_DEV].reset_index(drop=True)
assert set(det_train.source_id.astype(str)).isdisjoint(set(det_dev.source_id.astype(str)))
print('detector-train:',len(det_train),'detector-dev:',len(det_dev))


In [ ]:
def random_order_for_plans(plans,source_id):
    bids=np.asarray([p.block_id for p in plans],dtype=int)
    digest=hashlib.sha256(f'{seed}|{source_id}'.encode()).digest()
    rng=np.random.default_rng(int.from_bytes(digest[:8],'little'))
    out=bids.copy(); rng.shuffle(out)
    return out

def make_random_pairs(frame,bpp,label):
    covers=[]; stegos=[]; ids=[]; skipped=[]
    for j,row in frame.iterrows():
        sid=str(row.source_id); x=read_gray(row.path)
        plans=analyze_blocks(x,bs)
        order=random_order_for_plans(plans,sid)
        rr=run_frozen_image_precomputed(
            x,sid,bpp,'random',{'random':order},[],bs,seed,False,None,plans=plans
        )
        if rr['feasible']:
            if not (rr['exact_image'] and rr['exact_message'] and rr['ber']==0.0):
                raise RuntimeError(f'Reversibility invariant failed: {label} source {sid}')
            covers.append(x); stegos.append(rr['stego']); ids.append(sid)
        else:
            skipped.append(sid)
        if (j+1)%250==0:
            print(label,j+1,'/',len(frame),'feasible',len(covers))
    return covers,stegos,ids,skipped

train_c,train_s,train_ids,train_skip=make_random_pairs(det_train,teacher_bpp,'det-train')
dev_c,dev_s,dev_ids,dev_skip=make_random_pairs(det_dev,teacher_bpp,'det-dev')
print('train feasibility:',len(train_c),'/',len(det_train),len(train_c)/len(det_train))
print('dev feasibility:',len(dev_c),'/',len(det_dev),len(dev_c)/len(det_dev))
if len(train_c)<1000 or len(dev_c)<300:
    raise RuntimeError('Too few feasible pairs for repaired-detector development.')


## B. Train repaired residual CNN

The front end receives gray levels in their original 0–255 units before fixed high-pass filtering. Cover and stego members of a pair receive the same random dihedral augmentation during training. The detector never receives SRM-teacher scores or local-risk values.


In [ ]:
cnn,history,device=train_enhanced_residual_cnn(
    train_c,train_s,
    validation=(dev_c,dev_s),
    epochs=EPOCHS,
    pair_batch_size=PAIR_BATCH_SIZE,
    lr=3e-4,
    weight_decay=1e-4,
    seed=seed+8100,
    fixed_fpr=fixed_fpr,
)

hist=pd.DataFrame(history)
hist.to_csv(out/'enhanced_cnn_train_history.csv',index=False)
display(hist)
print('Device:',device)

save_enhanced_cnn(cnn,models/'enhanced_residual_cnn_05e.pt',{
    'architecture':'EnhancedResidualCNN-project-specific',
    'payload_bpp':teacher_bpp,
    'allocation':'random',
    'train_split_only':True,
    'detector_train_pairs':len(train_c),
    'detector_dev_pairs':len(dev_c),
    'epochs_fixed':EPOCHS,
    'pair_batch_size':PAIR_BATCH_SIZE,
    'lr':3e-4,
    'weight_decay':1e-4,
    'seed':seed+8100,
    'no_best_epoch_selection':True,
})


In [ ]:
dev_cover_scores=score_enhanced_residual_cnn(cnn,dev_c,device=device,batch_size=16)
dev_stego_scores=score_enhanced_residual_cnn(cnn,dev_s,device=device,batch_size=16)
y=np.tile([0,1],len(dev_c))
scores=np.column_stack([dev_cover_scores,dev_stego_scores]).reshape(-1)
dev_metrics=detector_metrics(y,scores,fixed_fpr)
dev_ci=paired_detector_bootstrap(
    dev_cover_scores,dev_stego_scores,fixed_fpr=fixed_fpr,
    n_resamples=max(n_boot,3000),confidence=confidence,seed=seed+8200,
)

sanity_pass=bool(dev_metrics['auc'] >= SANITY_AUC_THRESHOLD)
sanity={
    'sanity_pass':sanity_pass,
    'sanity_auc_threshold':SANITY_AUC_THRESHOLD,
    'payload_bpp':teacher_bpp,
    'dev_pairs':len(dev_c),
    'auc':float(dev_metrics['auc']),
    'auc_ci_low':float(dev_ci['auc_low']),
    'auc_ci_high':float(dev_ci['auc_high']),
    'tpr_at_5pct_fpr':float(dev_metrics['tpr_at_fpr']),
    'tpr_ci_low':float(dev_ci['tpr_low']),
    'tpr_ci_high':float(dev_ci['tpr_high']),
    'cover_score_mean':float(dev_cover_scores.mean()),
    'cover_score_std':float(dev_cover_scores.std()),
    'stego_score_mean':float(dev_stego_scores.mean()),
    'stego_score_std':float(dev_stego_scores.std()),
    'test_split_used':False,
}
(out/'detector_sanity.json').write_text(json.dumps(sanity,indent=2),encoding='utf-8')
print(json.dumps(sanity,indent=2))

fig,ax=plt.subplots(figsize=(6.2,4.2))
ax.plot(hist.epoch,hist.train_loss,marker='o',label='train loss')
ax.set_xlabel('Epoch'); ax.set_ylabel('BCE loss'); ax.set_title('Enhanced residual CNN training')
ax.grid(True,alpha=.2); fig.tight_layout(); fig.savefig(out/'enhanced_cnn_train_loss.png',dpi=300); plt.show()

fig,ax=plt.subplots(figsize=(6.2,4.2))
ax.plot(hist.epoch,hist.dev_auc,marker='o')
ax.axhline(SANITY_AUC_THRESHOLD,linewidth=1)
ax.axhline(0.5,linewidth=1)
ax.set_xlabel('Epoch'); ax.set_ylabel('Detector-dev ROC-AUC')
ax.set_title('Detector sanity trajectory (diagnostic only)')
ax.grid(True,alpha=.2); fig.tight_layout(); fig.savefig(out/'enhanced_cnn_dev_auc.png',dpi=300); plt.show()

if not sanity_pass:
    raise RuntimeError(
        f'Enhanced detector failed the pre-specified sanity threshold: AUC={dev_metrics["auc"]:.3f} < {SANITY_AUC_THRESHOLD:.2f}. '
        'Do not run transfer or notebook 06; redesign the independent detector.'
    )
print('SANITY PASSED — proceeding to the locked transfer comparison.')


## C. Locked transfer comparison

This section executes **only after detector sanity passes**. It reuses the exact source IDs that were common-feasible in `05d` and compares only the already pre-specified candidate \(\alpha=0.25\) with predictability-only \(\alpha=1\). No new alpha is chosen from the repaired-CNN result.


In [ ]:
meta05d=pd.read_csv('/workspace/results/cnn_transfer/transfer_common_feasible_metadata.csv')
locked_ids=list(dict.fromkeys(meta05d.source_id.astype(str).tolist()))
print('Locked 05d common-feasible source IDs:',len(locked_ids))
if len(locked_ids)<300:
    raise RuntimeError('05d common-feasible transfer metadata is missing or unexpectedly small.')

val_by_id={str(r.source_id):r for _,r in val.iterrows()}
missing=[sid for sid in locked_ids if sid not in val_by_id]
if missing:
    raise RuntimeError(f'{len(missing)} locked 05d IDs are missing from current validation manifest.')


In [ ]:
def make_orders(block_rows,source_id,alpha):
    bids=np.asarray([r['block_id'] for r in block_rows],int)
    p=np.asarray([r['predictability'] for r in block_rows],float)
    d=np.asarray([r['detectability_risk'] for r in block_rows],float)
    digest=hashlib.sha256(f'{seed}|{source_id}'.encode()).digest()
    rng=np.random.default_rng(int.from_bytes(digest[:8],'little'))
    rnd=bids.copy(); rng.shuffle(rnd)
    return {
        'raster':bids.copy(),
        'random':rnd,
        'predictability':bids[np.argsort(-p,kind='stable')],
        'detectability':bids[np.argsort(d,kind='stable')],
        'joint':bids[rank_blocks(p,d,alpha,1.0-alpha)],
    }

transfer_c=[]; transfer_candidate=[]; transfer_reference=[]; transfer_rows=[]
unexpected_infeasible=[]
for j,sid in enumerate(locked_ids):
    row=val_by_id[sid]; x=read_gray(row.path)
    br=local_risk.score_image_blocks(x,sid)
    plans=analyze_blocks(x,bs)
    per={}
    for a in [candidate_alpha,reference_alpha]:
        rr=run_frozen_image_precomputed(
            x,sid,teacher_bpp,'joint',make_orders(br,sid,a),br,bs,seed,False,None,plans=plans
        )
        if not rr['feasible']:
            unexpected_infeasible.append((sid,a)); per={}; break
        if not (rr['exact_image'] and rr['exact_message'] and rr['ber']==0.0):
            raise RuntimeError(f'Reversibility invariant failed for {sid}, alpha={a}')
        per[a]=rr
    if not per:
        continue
    transfer_c.append(x)
    transfer_candidate.append(per[candidate_alpha]['stego'])
    transfer_reference.append(per[reference_alpha]['stego'])
    for a in [candidate_alpha,reference_alpha]:
        rr=per[a]
        transfer_rows.append({
            'source_id':sid,'alpha':a,'psnr':rr['psnr'],'ssim':rr['ssim'],
            'used_blocks':rr['used_blocks'],
            'selected_P_mean':rr['selected_predictability_mean'],
            'selected_D_mean':rr['selected_detectability_risk_mean'],
        })
    if (j+1)%50==0:
        print('locked transfer',j+1,'/',len(locked_ids),'complete',len(transfer_c))

print('Transfer pairs regenerated:',len(transfer_c),'/',len(locked_ids))
print('Unexpected infeasible cases:',len(unexpected_infeasible))
if len(transfer_c) < 0.98*len(locked_ids):
    raise RuntimeError('Too many locked 05d cases became infeasible; investigate before inference.')
pd.DataFrame(transfer_rows).to_csv(out/'locked_transfer_metadata.csv',index=False)


In [ ]:
cover_scores=score_enhanced_residual_cnn(cnn,transfer_c,device=device,batch_size=16)
candidate_scores=score_enhanced_residual_cnn(cnn,transfer_candidate,device=device,batch_size=16)
reference_scores=score_enhanced_residual_cnn(cnn,transfer_reference,device=device,batch_size=16)

def method_summary(stego_scores,alpha,seed_offset):
    y=np.tile([0,1],len(cover_scores))
    sc=np.column_stack([cover_scores,stego_scores]).reshape(-1)
    m=detector_metrics(y,sc,fixed_fpr)
    ci=paired_detector_bootstrap(
        cover_scores,stego_scores,fixed_fpr=fixed_fpr,
        n_resamples=max(n_boot,3000),confidence=confidence,seed=seed+seed_offset,
    )
    return {
        'alpha':alpha,'n_pairs':len(cover_scores),'auc':m['auc'],
        'auc_ci_low':ci['auc_low'],'auc_ci_high':ci['auc_high'],
        'tpr_at_5pct_fpr':m['tpr_at_fpr'],'tpr_ci_low':ci['tpr_low'],'tpr_ci_high':ci['tpr_high'],
        'cover_score_mean':float(cover_scores.mean()),'stego_score_mean':float(np.mean(stego_scores)),
        'paired_score_delta_mean':float(np.mean(stego_scores-cover_scores)),
        'paired_score_delta_median':float(np.median(stego_scores-cover_scores)),
    }

summary=pd.DataFrame([
    method_summary(candidate_scores,candidate_alpha,8400),
    method_summary(reference_scores,reference_alpha,8500),
])
summary.to_csv(out/'enhanced_cnn_transfer_summary.csv',index=False)
display(summary)

paired=paired_method_bootstrap(
    cover_scores,candidate_scores,reference_scores,
    fixed_fpr=fixed_fpr,n_resamples=max(n_boot,5000),confidence=confidence,seed=seed+8600,
)
print(json.dumps(paired,indent=2))


In [ ]:
primary_pass=bool(paired['delta_mean_diff_high'] < 0.0)
point_direction=bool(paired['delta_mean_diff'] < 0.0)
if primary_pass:
    decision='TRANSFER_CONFIRMED_PRIMARY_ENDPOINT'
elif point_direction:
    decision='TRANSFER_SIGNAL_NOT_CONCLUSIVE'
else:
    decision='TRANSFER_NOT_CONFIRMED'

decision_obj={
    'decision':decision,
    'detector_sanity_pass':sanity_pass,
    'detector_dev_auc':float(dev_metrics['auc']),
    'detector_dev_auc_ci_low':float(dev_ci['auc_low']),
    'detector_dev_auc_ci_high':float(dev_ci['auc_high']),
    'candidate_alpha':candidate_alpha,
    'reference_alpha':reference_alpha,
    'payload_bpp':teacher_bpp,
    'primary_endpoint':'paired enhanced-CNN score-change difference candidate minus alpha=1.0',
    'primary_difference':float(paired['delta_mean_diff']),
    'primary_ci_low':float(paired['delta_mean_diff_low']),
    'primary_ci_high':float(paired['delta_mean_diff_high']),
    'auc_diff':float(paired['auc_diff']),
    'auc_diff_low':float(paired['auc_diff_low']),
    'auc_diff_high':float(paired['auc_diff_high']),
    'tpr_diff':float(paired['tpr_diff']),
    'tpr_diff_low':float(paired['tpr_diff_low']),
    'tpr_diff_high':float(paired['tpr_diff_high']),
    'transfer_pairs':len(cover_scores),
    'source_ids_locked_from_05d':True,
    'test_split_used':False,
    'frozen_allocator_modified':False,
}
(out/'decision.json').write_text(json.dumps(decision_obj,indent=2),encoding='utf-8')
print(json.dumps(decision_obj,indent=2))

fig,ax=plt.subplots(figsize=(6.2,4.2))
ax.bar(['alpha=0.25','alpha=1.0'],summary.auc.values)
ax.axhline(0.5,linewidth=1)
ax.set_ylabel('Enhanced-CNN ROC-AUC'); ax.set_title('Locked independent transfer comparison')
fig.tight_layout(); fig.savefig(out/'locked_transfer_auc.png',dpi=300); plt.show()

if primary_pass:
    print('\nNEXT: repaired independent detector passed sanity and confirmed the locked transfer endpoint.')
    print('Review outputs, then freeze alpha in a separate notebook before touching the test split.')
elif point_direction:
    print('\nNEXT: repaired detector has power, but transfer direction is not conclusive. Do not freeze yet.')
else:
    print('\nNEXT: repaired detector has power, but transfer is not confirmed. Do not freeze the allocator.')


## Outputs

Saved under `/workspace/results/cnn_repair_transfer/`:

- `enhanced_cnn_train_history.csv`
- `detector_sanity.json`
- `enhanced_cnn_train_loss.png`
- `enhanced_cnn_dev_auc.png`
- `locked_transfer_metadata.csv`
- `enhanced_cnn_transfer_summary.csv`
- `locked_transfer_auc.png`
- `decision.json`

The model is saved as `/workspace/results/models/enhanced_residual_cnn_05e.pt`.

**Do not run notebook 06 unless the repaired detector passes sanity and the transfer decision is reviewed.**
